In [1]:
from tqdm import tqdm
tqdm.pandas()
from glob import glob

import json
import csv
import numpy as np
import numpy.random as rand
import pandas as pd
from collections import Counter

import rdflib
from rdflib import Graph
from data.data import CollectionAccessor, ImageHandler, EmbeddingSpaceAccessor

from search import Search, Randomiser, Equaliser, GraphSearcher, EmbeddingSearcher, TextEmbeddingSearcher
from moon import MOON, Moon

import requests as rq
import matplotlib.pyplot as plt

In [ ]:
from app import init_DMG, init_MKG, order_collection

def lifespan():
    global DMG
    global DMG_searcher
    global DMG_concept_search
    DMG, DMG_searcher, DMG_concept_search = init_DMG()
lifespan()

In [2]:
prefix = "http://0.0.0.0:8080/DMG_2025-06-06/"

params = dict(object_ids_index_of="1990-0033_7-9,1408", object_ids="1976-0043",
    k=9, lat=51.05, long=3.71, model_ids="KGSearcher0,SemanticSearcher2,VisualSearcher3",
    filter_text="e", skip=0, limit=9)

In [3]:
search_res = rq.get(prefix+"search", params=params).json()
search_res = pd.Series(search_res)

order_res = rq.get(prefix+"search/order", params=params).json()
order_res = pd.DataFrame.from_records(order_res).set_index("inventory_number")

indexes = rq.get(prefix+"search/order/indexof", params=params).json()
indexes = pd.Series(indexes)

In [ ]:
search_res.sort_values().iloc[::-1]

In [ ]:
order_res.sort_values(by="order_index").order_index

In [ ]:
indexes

In [ ]:
sample = rq.get(prefix+"search/sample", params=params).json()
orig_recs = pd.DataFrame.from_records(sample["original_records"]).set_index("inventory_number")
sample = pd.DataFrame.from_records(sample["sampled_records"]).set_index("inventory_number")

In [ ]:
sample.order_index

In [ ]:
order_res.loc[sample.index].order_index

In [ ]:
request_ids = ",".join(sample.index)
print(request_ids)
params["object_ids_index_of"] = request_ids
indexes = rq.get(prefix+"search/order/indexof", 
                 params=params).json()
indexes = pd.Series(indexes)

indexes

In [ ]:
order_res.loc[sample.index].order_index

In [ ]:
search_res.argsort().loc[sample.index]

---

## new `search/sample`

In [ ]:
# scores = search_collection(collection_id, object_ids, concept, model_ids)
scores = search_res
order_index = scores.argsort()

rand_recs = DMG_searcher.sample(DMG, scores=scores, size=9)#, temp=moon_force, size=k)
sample_order_index = order_index.loc[rand_recs.index]

original_recs = DMG.loc[["1976-0043"]]
original_order_index = order_index.loc[original_recs.index]
    


In [ ]:
scores.sort_values(ascending=False)

score_index = pd.Series(range(len(scores)), index=scores.sort_values(ascending=False).index)
score_index.loc[rand_recs.index]

In [ ]:
order_res.loc[rand_recs.index].order_index

In [ ]:
plt.plot(order_index.sort_index().values, order_res.sort_index().order_index, ".")

In [ ]:
order_res.sort_values(by="order_index").order_index